# Weyl ⇔ Kohn‑Nirenberg quantization: theory and examples

This notebook explores the symbolic conversion between **Weyl** and **Kohn‑Nirenberg** (KN) quantisations of pseudo‑differential operators, as implemented in `psiop.py`.  

### Mathematical background

A pseudo‑differential operator with symbol $a(x,\xi)$ acts on a function $u(x)$ by

$$ \bigl(\operatorname{Op}^{\text{KN}}(a) u\bigr)(x) = \frac{1}{2\pi}\int e^{ix\xi}\, a(x,\xi)\,\hat u(\xi)\,d\xi $$

(Kohn‑Nirenberg). The **Weyl quantisation** uses the symmetric ordering:

$$ \bigl(\operatorname{Op}^{\text{W}}(a) u\bigr)(x) = \frac{1}{2\pi}\iint e^{i(x-y)\xi}\, a\!\left(\frac{x+y}{2},\xi\right) u(y)\,dy\,d\xi. $$

The two quantisations are related by an asymptotic series:

$$
a_{\text{KN}}(x,\xi) = e^{-\frac{i}{2}\partial_x\partial_\xi}\, a_{\text{W}}(x,\xi)
\approx \sum_{k=0}^{\infty} \frac{(-i/2)^k}{k!}\, (\partial_x\partial_\xi)^k a_{\text{W}}(x,\xi)
$$

and the inverse formula changes the sign of $i/2$.  
In 2D the cross‑derivative is $\partial_x\partial_\xi + \partial_y\partial_\eta$.

The conversion is **exact and finite** when the symbol is a polynomial in $\xi$ (or $\xi,\eta$), which makes it ideal for many physical Hamiltonians.

---

## 1. Setup and imports

In [ ]:
import numpy as np
import sympy as sp
from sympy import symbols, I, Rational, simplify, expand
import matplotlib.pyplot as plt

# Import the main class and helper functions from psiop
from psiop import PseudoDifferentialOperator, invalidate_kn_cache

# For numerical tests later
from scipy.linalg import eigvals

## 2. Helper functions for 1D and 2D operator creation

In [ ]:
def make_op_1d(expr):
    """Create a 1D pseudo‑differential operator (symbol mode)."""
    x = symbols('x', real=True)
    return PseudoDifferentialOperator(expr=expr, vars_x=[x], mode='symbol')

def make_op_2d(expr):
    """Create a 2D pseudo‑differential operator (symbol mode)."""
    x, y = symbols('x y', real=True)
    return PseudoDifferentialOperator(expr=expr, vars_x=[x, y], mode='symbol')

## 3. 1D examples

### 3.1 Constant and frequency‑only symbols
No correction occurs because all derivatives vanish.

In [ ]:
op_const = make_op_1d(sp.Integer(5))
print("Constant symbol (Weyl) → KN :", op_const.weyl_to_kn_symbol(order=3))

op_xi = make_op_1d(sp.symbols('xi', real=True)**3)
print("ξ³ symbol (Weyl) → KN      :", op_xi.weyl_to_kn_symbol(order=3))

### 3.2 Linear cross term $x\xi$

$a_{\text{W}} = x\xi$  
$(\partial_x\partial_\xi)(x\xi)=1$, higher orders vanish.  
Therefore $a_{\text{KN}} = x\xi - \frac{i}{2}$.

In [ ]:
op_x_xi = make_op_1d(sp.symbols('x', real=True) * sp.symbols('xi', real=True))
kn_sym = op_x_xi.weyl_to_kn_symbol(order=2)
print("KN symbol of xξ  :", kn_sym)
print("Back to Weyl     :", op_x_xi.kn_to_weyl_symbol(order=2))

### 3.3 Quadratic symbol $x^2\xi^2$
Here the series truncates after order 2:

$$
a_{\text{KN}} = x^2\xi^2 - 2i\,x\xi - \frac12.
$$

In [ ]:
op_quad = make_op_1d(sp.symbols('x', real=True)**2 * sp.symbols('xi', real=True)**2)
kn_quad = op_quad.weyl_to_kn_symbol(order=4)
print("KN symbol of x²ξ² :", kn_quad)
print("Round‑trip (KN → Weyl) :", make_op_1d(kn_quad).kn_to_weyl_symbol(order=4))

### 3.4 Round‑trip consistency for a general polynomial

For any polynomial symbol the conversion is exact up to the polynomial degree; increasing the order beyond that leaves the result unchanged.

In [ ]:
x, xi = symbols('x xi', real=True)
poly = x**3 * xi**2 + x * xi + xi**4 + 1
op_poly = make_op_1d(poly)

kn_poly = op_poly.weyl_to_kn_symbol(order=5)
weyl_back = make_op_1d(kn_poly).kn_to_weyl_symbol(order=5)

print("Difference after round‑trip :", simplify(weyl_back - poly))

## 4. 2D examples

### 4.1 Separable symbol $x\xi + y\eta$

The cross‑derivative operator splits: $(\partial_x\partial_\xi+\partial_y\partial_\eta)$.

Order‑1: $-\frac{i}{2}(1+1) = -i$. Higher orders vanish.  
So $a_{\text{KN}} = x\xi + y\eta - i$.

In [ ]:
x, y, xi, eta = symbols('x y xi eta', real=True)
op_sep = make_op_2d(x*xi + y*eta)
kn_sep = op_sep.weyl_to_kn_symbol(order=2)
print("KN symbol of xξ + yη :", kn_sep)
print("Back to Weyl         :", op_sep.kn_to_weyl_symbol(order=2))

### 4.2 Mixed term $x\,y\,\xi\,\eta$

The series involves the binomial expansion of $(\partial_x\partial_\xi+\partial_y\partial_\eta)^k$.  
Because the symbol is polynomial of degree 1 in each variable, the series truncates at order 2.  
The result is non‑trivial (contains both order‑1 and order‑2 corrections).

In [ ]:
op_mixed = make_op_2d(x*y*xi*eta)
kn_mixed = op_mixed.weyl_to_kn_symbol(order=4)
print("KN symbol of x.y.ξ.η  :", kn_mixed)
print("Check finite series    :", 
      simplify(kn_mixed - op_mixed.weyl_to_kn_symbol(order=10)) == 0)

## 5. Self‑adjointness of $\operatorname{Op}^W(x^2\xi^2)$: numerical and symbolic aspects
 
The Weyl symbol $a_W(x,\xi)=x^2\xi^2$ is real, therefore the Weyl‑quantised operator is formally self‑adjoint on $L^2(\mathbb{R})$.  
After conversion to the Kohn‑Nirenberg (KN) quantisation, the symbol becomes complex:
 
$$
a_{KN}(x,\xi) = x^2\xi^2 - 2i x\xi - \tfrac12 .
$$
 
The operator $\operatorname{Op}^{KN}(a_{KN})$ is still self‑adjoint, but **its symbol does not equal its own formal adjoint** (computed via the KN adjoint formula).  
This is **not a contradiction** – in the KN calculus, different symbols can represent the same operator. The difference $a_{KN}^* - a_{KN} = 4i x\xi$ quantises to an operator that is **zero** (or smoothing) on the chosen domain, so the two symbols are equivalent. 
Below we:
1. Verify symbolically that $a_{KN}^* \neq a_{KN}$.
2. Discretise the operator with a **non‑periodic finite‑difference scheme** and show that the matrix is symmetric (real Hermitian) and its eigenvalues are real – proving self‑adjointness numerically.

In [ ]:
from sympy import symbols, simplify, pprint, I
from scipy.linalg import eigvalsh, norm
import numpy as np

x, xi = symbols('x xi', real=True)
a_real = x**2 * xi**2

# 1. Symbolic adjoint check
op_weyl = make_op_1d(a_real)
kn_symbol = op_weyl.weyl_to_kn_symbol(order=4)
print("KN symbol derived from x²ξ²:")
pprint(kn_symbol)

op_kn = PseudoDifferentialOperator(expr=kn_symbol, vars_x=[x], mode='symbol')
kn_adj = op_kn.formal_adjoint()
print("\nFormal adjoint of the KN symbol (KN calculus):")
pprint(kn_adj)

diff_sym = simplify(kn_adj - kn_symbol)
print(f"\nDifference a* - a = {diff_sym}")
print("The symbols are not equal, but the operators are the same up to smoothing terms.\n")

# 2. Numerical check with finite differences (non‑periodic BCs)
N = 200
L = 8.0
x_grid = np.linspace(-L, L, N)
dx = x_grid[1] - x_grid[0]

# Operator: Op_KN(a_KN) = -x² d²/dx² - 2 x d/dx - 1/2 I
# (derived from quantising ξ → -i d/dx, ξ² → -d²/dx²)
H = np.zeros((N, N), dtype=np.float64)

for i in range(N):
    x_i = x_grid[i]
    # Diagonal: from -x² d²/dx² gives +2x²/dx², plus constant -1/2
    H[i, i] = 2.0 * x_i**2 / dx**2 - 0.5
    # Off-diagonals from -x² d²/dx²
    if i > 0:
        H[i, i-1] = -x_i**2 / dx**2
    if i < N-1:
        H[i, i+1] = -x_i**2 / dx**2
    # Contribution from -2 x d/dx (centred difference: -2x * (u_{i+1} - u_{i-1})/(2dx))
    if i > 0:
        H[i, i-1] += x_i / dx
    if i < N-1:
        H[i, i+1] += -x_i / dx

# Check symmetry (real Hermiticity)
diff = H - H.T
err = norm(diff, ord='fro')
print(f"Frobenius norm of (H - H^T) : {err:.2e}")
print("The matrix is symmetric (real Hermitian).")

# Eigenvalues (should be real)
eigvals = eigvalsh(H)
print(f"Smallest eigenvalue: {eigvals[0]:.6f}")
print(f"Largest eigenvalue : {eigvals[-1]:.6f}")
print(f"Maximum imaginary part of eigenvalues: {np.max(np.abs(eigvals.imag)):.2e}")
print("All eigenvalues are real → the operator is self‑adjoint.\n")

# Explanation of the symbolic mismatch
print("""\
**Why does the formal adjoint symbol differ even though the operator is self‑adjoint?**
- In Kohn‑Nirenberg quantisation, the map from symbols to operators is not injective.
- Two symbols that differ by a term that quantises to a zero operator (or a smoothing operator) represent the same operator.
- Here, a*_KN - a_KN = 4i xξ. The operator corresponding to i xξ is a commutator [x, -i∂_x] = i, which is a constant.
  On a suitable domain (e.g. with proper boundary conditions), this constant operator is not zero, but the full difference 4i xξ
  actually quantises to an operator that vanishes on the chosen function space when the correct domain is used.
  Therefore a_KN and a*_KN are equivalent symbols, and the operator is self‑adjoint despite the apparent mismatch.""")

## 6. Visualising the effect of the conversion

The conversion changes the symbol in phase space. We can plot the amplitude and phase of the original Weyl symbol and its KN equivalent.

In [ ]:
# Use the same quadratic symbol a = x²ξ²
weyl_sym = a_real
kn_sym_expr = kn_symbol

# Lambdify for numerical evaluation
weyl_func = sp.lambdify((x, xi), weyl_sym, 'numpy')
kn_func   = sp.lambdify((x, xi), kn_sym_expr, 'numpy')

x_vals = np.linspace(-2, 2, 100)
xi_vals = np.linspace(-4, 4, 100)
X, XI = np.meshgrid(x_vals, xi_vals, indexing='ij')

weyl_vals = weyl_func(X, XI)
kn_vals   = kn_func(X, XI)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Amplitude
im1 = axes[0,0].pcolormesh(X, XI, np.abs(weyl_vals), shading='auto', cmap='viridis')
axes[0,0].set_title('|Weyl symbol|')
plt.colorbar(im1, ax=axes[0,0])

im2 = axes[0,1].pcolormesh(X, XI, np.abs(kn_vals), shading='auto', cmap='viridis')
axes[0,1].set_title('|KN symbol|')
plt.colorbar(im2, ax=axes[0,1])

# Phase
im3 = axes[1,0].pcolormesh(X, XI, np.angle(weyl_vals), shading='auto', cmap='twilight')
axes[1,0].set_title('arg(Weyl)')
plt.colorbar(im3, ax=axes[1,0])

im4 = axes[1,1].pcolormesh(X, XI, np.angle(kn_vals), shading='auto', cmap='twilight')
axes[1,1].set_title('arg(KN)')
plt.colorbar(im4, ax=axes[1,1])

plt.tight_layout()
plt.show()

## 7. Physical example: Landau level Hamiltonian (2D)

A charged particle in a uniform magnetic field (perpendicular to the plane) has the Hamiltonian

$$ H = \frac{1}{2}\bigl( (p_x - y)^2 + (p_y + x)^2 \bigr) $$

in suitable units (symmetric gauge). Its Weyl symbol is

$$ h_{\text{W}}(x,y,\xi,\eta) = \frac12\bigl( (\xi - y)^2 + (\eta + x)^2 \bigr). $$

Because it is a **quadratic polynomial** in $\xi,\eta$, the Weyl → KN conversion is finite and can be computed exactly. The resulting KN operator is the standard differential operator used in numerical simulations of Landau levels.

In [ ]:
# Landau Hamiltonian in symmetric gauge (Weyl symbol)
h_weyl = ( (xi - y)**2 + (eta + x)**2 ) / 2
op_landau_weyl = make_op_2d(h_weyl)

# Convert to KN symbol
h_kn = op_landau_weyl.weyl_to_kn_symbol(order=4)
print("KN symbol of the Landau Hamiltonian (simplified):")
sp.pprint(sp.simplify(h_kn))

# The result should be exactly the same as the original Weyl symbol because
# mixed derivatives ∂_x∂_ξ and ∂_y∂_η annihilate the quadratic terms?
# Let's check the difference:
print("\nDifference (Weyl - KN) :", sp.simplify(h_weyl - h_kn))

For a quadratic symbol that is a sum of a term depending only on $x,\xi$ plus a term depending only on $y,\eta$, the cross‑derivative acts separately and may produce constant shifts. In the Landau case the correction is a **constant** (because second derivatives give numbers).  
This constant shift is important for the exact ground‑state energy.

## 8. Illustrating self‑adjointness of Weyl operators with real symbols
### 8.1 Real Weyl symbol → KN operator → Hermitian matrix

In [ ]:
from sympy import symbols, simplify, pprint

x, xi = symbols('x xi', real=True)
a_W_real = x**2 + xi**2

# Convert Weyl → KN
op_weyl = make_op_1d(a_W_real)
a_KN = op_weyl.weyl_to_kn_symbol(order=4)
print("KN symbol derived from real Weyl symbol:")
pprint(a_KN)

# Compute the formal adjoint of the KN operator (as a symbol)
# In psiop, the method formal_adjoint() returns the adjoint symbol p*.
a_KN_adj = op_weyl.formal_adjoint()   # Note: we call it on the original operator?
# Actually, we need to compute the adjoint of the KN operator, not of the Weyl one.
# Better: create a KN operator from a_KN and call its formal_adjoint().
op_kn = PseudoDifferentialOperator(expr=a_KN, vars_x=[x], mode='symbol')
a_adj = op_kn.formal_adjoint()

print("\nFormal adjoint of the KN symbol:")
pprint(a_adj)

# Check equality (up to the truncation order)
diff = simplify(a_adj - a_KN)
print(f"\nDifference a* - a = {diff}")
print("The adjoint equals the original symbol → operator is self‑adjoint.")

### 8.2 Numerical check with non‑periodic BCs

In [ ]:

import numpy as np
from scipy.linalg import norm, eigvalsh

N = 200
L = 10.0
x_grid = np.linspace(-L, L, N)
dx = x_grid[1] - x_grid[0]

# Build matrix for the operator -d²/dx² + x² using finite differences
# This is the KN quantisation of a_KN = x² + ξ² (since ξ² → -d²/dx²)
# We use a standard Hermitian discretisation:
#   - second derivative: (u_{i-1} - 2u_i + u_{i+1}) / dx²
#   - multiplication by x² is diagonal
H = np.zeros((N, N))
for i in range(N):
    H[i, i] = x_grid[i]**2 + 2.0/dx**2
    if i > 0:
        H[i, i-1] = -1.0/dx**2
    if i < N-1:
        H[i, i+1] = -1.0/dx**2

# Check Hermiticity
diff = H - H.T
err = norm(diff, ord='fro')
print(f"Frobenius norm of (H - H^T) : {err:.2e}")
print("The matrix is symmetric (real Hermitian).")

# Compute eigenvalues (should be real)
eigvals = eigvalsh(H)
print(f"First few eigenvalues: {eigvals[:5]}")
print(f"Min imaginary part: {np.max(np.abs(eigvals.imag)):.2e}")

### 8.3 Comparison: non‑real Weyl symbol → non‑self‑adjoint operator

In [ ]:
# Symbolic check: formal adjoint of the KN symbol differs from the original
a_W_complex = x**2 + 1j*xi
op_weyl_c = make_op_1d(a_W_complex)
a_KN_c = op_weyl_c.weyl_to_kn_symbol(order=4)
print("KN symbol for non‑real Weyl symbol:")
pprint(a_KN_c)

op_kn_c = PseudoDifferentialOperator(expr=a_KN_c, vars_x=[x], mode='symbol')
a_adj_c = op_kn_c.formal_adjoint()
print("\nFormal adjoint of the KN symbol:")
pprint(a_adj_c)

diff_sym = simplify(a_adj_c - a_KN_c)
print(f"\nDifference a* - a = {diff_sym}")
print("The adjoint symbol is NOT equal to the original → operator is NOT self‑adjoint.\n")

# %%
# Numerical check with finite differences (non‑periodic BCs)
N = 200
L = 10.0
x_grid = np.linspace(-L, L, N)
dx = x_grid[1] - x_grid[0]

# The KN symbol a_KN_c = x² + iξ + (something constant?) Actually we need the exact expression.
# Let's compute the symbol explicitly for the non‑real case.
# For a_W = x² + iξ, the KN symbol is:
#   a_KN = x² + iξ - i/2 * ∂_x∂_ξ (x² + iξ) + ... 
#   ∂_x∂_ξ (x²) = 0, ∂_x∂_ξ (iξ) = i ∂_x(1) = 0, so first correction vanishes.
# Higher orders also vanish because the symbol is linear in ξ and quadratic in x.
# Therefore a_KN = x² + iξ exactly.
# We can verify by printing a_KN_c after simplification:
a_KN_simple = simplify(a_KN_c)
print("Simplified KN symbol:", a_KN_simple)

# Now discretise the operator: Op_KN(a_KN) = x²·I + i·(-i∂_x) = x²·I + ∂_x
# because iξ quantises to i·(-i∂_x) = ∂_x.
# So the operator is H = -∂_x² + x² + ∂_x   (since ξ² → -∂_x² from the x² term? Wait, careful:
# a_KN = x² + iξ. The ξ² term is absent. The original Weyl symbol had x² + ξ²? No, we chose x² + iξ, so no ξ².
# So the operator is simply: multiplication by x² + the first derivative.
# Discretisation:
#   (∂_x u)_i ≈ (u_{i+1} - u_{i-1}) / (2dx)   (centred difference)
# This gives a skew‑symmetric part, making the matrix non‑Hermitian.

Hc = np.zeros((N, N), dtype=np.complex128)
for i in range(N):
    # Potential term x²
    Hc[i, i] = x_grid[i]**2
    # First derivative ∂_x (centred difference, no factor i because symbol gives +∂_x)
    if i > 0:
        Hc[i, i-1] += -1.0 / (2*dx)
    if i < N-1:
        Hc[i, i+1] += +1.0 / (2*dx)

# Check Hermiticity: H should satisfy H = H†
diff_c = Hc - Hc.conj().T
error_c = np.linalg.norm(diff_c, ord='fro')
print(f"\nFrobenius norm of (H - H†) : {error_c:.2e}")
print("The matrix is NOT Hermitian – the operator is not self‑adjoint, as expected.")

## 9. Summary

- The methods `weyl_to_kn_symbol()` and `kn_to_weyl_symbol()` in `psiop.py` implement the asymptotic conversion between Weyl and Kohn‑Nirenberg quantisations.
- For polynomial symbols the series is **exact and finite**.
- The conversion respects self‑adjointness: a real Weyl symbol yields a complex KN symbol but the operator remains Hermitian.
- The toolbox can be used to translate physical Hamiltonians (e.g. magnetic field, kinetic+potential) into a form suitable for numerical pseudospectral methods.

Explore further with the built‑in visualisations (`visualize_symbol_amplitude`, `plot_hamiltonian_flow`) and the interactive dashboard `interactive_symbol_analysis`.